<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 05 · Storytelling Capstone
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The capstone should prove that the delegate can combine cleaning, summarizing,
charting, and narrative into one small project that could live in a portfolio.

## Setup
We make the notebook portable first so the capstone data and saved notes use
stable relative paths.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "2_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Capstone Orders Table
Every capstone should start by making the raw input visible. The first analysis
cell reads the file and prints the columns before any transformation begins.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

data_dir = BOOK_ROOT / 'data'
project_dir = BOOK_ROOT / 'artifacts' / 'module2_capstone'
project_dir.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(data_dir / 'capstone_orders.csv')
print(orders.head())
print()
print(orders.columns.tolist())


## Add Time Fields and Fill Gaps
We convert the order date, fill missing discounts, and derive month and weekday
columns so the table can support both summaries and charts.


In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['discount_pct'] = orders['discount_pct'].fillna(0)
orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)
orders['weekday'] = orders['order_date'].dt.day_name()

print(orders[['order_date', 'channel', 'country', 'amount', 'discount_pct']].head())


## Build the Core Summaries
The heart of the capstone is a small set of grouped tables. They provide the
numerical backbone of the story before any visual design is added.


In [ ]:
channel_summary = (
    orders.groupby('channel')['amount']
    .agg(total_amount='sum', average_amount='mean', order_count='size')
    .sort_values('total_amount', ascending=False)
)

country_summary = (
    orders.groupby('country')['amount']
    .agg(total_amount='sum', average_amount='mean', order_count='size')
    .sort_values('total_amount', ascending=False)
)

monthly_summary = (
    orders.groupby('order_month')['amount']
    .sum()
    .sort_index()
)

print('channel summary:')
print(channel_summary)
print()
print('country summary:')
print(country_summary)


## Turn the Story Into Charts
The plotting cells convert those summaries into a visual narrative. One chart
compares channels directly, and the second adds a time-based view.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
channel_summary['total_amount'].plot(kind='bar', ax=ax, color='#0B3C78')
ax.set_title('Total Amount by Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('Total Amount')
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
monthly_summary.plot(kind='line', marker='o', ax=ax, color='#0B3C78')
ax.set_title('Monthly Amounts')
ax.set_xlabel('Month')
ax.set_ylabel('Total Amount')
ax.grid(alpha=0.35, linestyle='--', linewidth=0.6)
fig.tight_layout()
plt.show()


## Package the Capstone Notes
The final cell writes a small README, checklist, and findings note so the
capstone already looks like something that could move into a portfolio repo.


In [ ]:
readme_path = project_dir / 'README.md'
checklist_path = project_dir / 'checklist.txt'
notes_path = project_dir / 'findings.txt'

readme_path.write_text(
    '# Module 2 Capstone\n\n'
    'Question: Which channels and countries contribute the most order amount?\n\n'
    'Dataset: data/capstone_orders.csv\n\n'
    'Run the notebook from the book root so relative paths stay stable.\n'
)
checklist_path.write_text(
    '- State one focused question.\n'
    '- Clean dates and missing values.\n'
    '- Build one numeric table and one chart.\n'
    '- Save the output to figures/.\n'
    '- Write a short summary in plain language.\n'
)
notes_path.write_text(
    'Total amount is concentrated in a few channels, and the month-by-month\n'
    'series gives a compact story for the project README.\n'
)

print(readme_path.resolve())
print(checklist_path.resolve())
print(notes_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">